# Competitor Intelligence Report Generator

Give this tool any company URL and it will:
1. Scrape the company's website to understand what they do
2. Use **Tavily** to find their top competitors in real time
3. Scrape each competitor's website
4. Use an LLM to extract structured profiles from all companies
5. Generate a full **Competitive Intelligence Report** including:
   - Company & competitor profiles
   - Side-by-side competitive matrix
   - Market standing analysis
   - Strategic recommendations

---

**Default models: `claude-haiku-4-5` for extraction, `claude-sonnet-4-5` for the report.**  
You can switch to any other provider by changing the model name in Cell 3.  
The wrapper automatically routes to the correct provider.


## Cell 1 — Check API Keys
Make sure your `.env` file is loaded and the keys you need are present.

In [1]:
import os
import sys
sys.path.insert(0, os.getcwd())

from dotenv import load_dotenv
load_dotenv(dotenv_path=os.path.join(os.getcwd(), '.env'), override=True)

keys_to_check = {
    "TAVILY_API_KEY":    "Tavily    (required for competitor search)",
    "ANTHROPIC_API_KEY": "Anthropic (default LLM — claude-haiku-4-5 / claude-sonnet-4-5)",
    "GROQ_API_KEY":      "Groq      (free alternative)",
    "GOOGLE_API_KEY":    "Gemini    (free alternative)",
    "OPENAI_API_KEY":    "OpenAI    (paid alternative)",
    "MISTRAL_API_KEY":   "Mistral   (free tier alternative)",
}

print("API Key Status:\n")
for env_var, label in keys_to_check.items():
    val = os.getenv(env_var, "")
    status = "OK" if val and "your_" not in val else "NOT SET"
    print(f"  {status:8s}  {label}")

API Key Status:

  OK        Tavily    (required for competitor search)
  OK        Anthropic (default LLM — claude-haiku-4-5 / claude-sonnet-4-5)
  OK        Groq      (free alternative)
  OK        Gemini    (free alternative)
  NOT SET   OpenAI    (paid alternative)
  NOT SET   Mistral   (free tier alternative)


## Cell 2 — Available Models
See every model you can use and which provider it routes to.

In [2]:
from config import MODEL_TO_PROVIDER, FAST_MODEL, SMART_MODEL

print("Available models:\n")
for model, provider in MODEL_TO_PROVIDER.items():
    tag = ""
    if model == FAST_MODEL:  tag = "  <-- default FAST model (extraction)"
    if model == SMART_MODEL: tag = "  <-- default SMART model (report)"
    print(f"  {provider:10s}  {model}{tag}")

Available models:

  groq        llama-3.3-70b-versatile
  groq        llama-3.1-8b-instant
  groq        mixtral-8x7b-32768
  groq        gemma2-9b-it
  gemini      gemini-2.0-flash
  gemini      gemini-1.5-flash
  gemini      gemini-1.5-pro
  openai      gpt-4o
  openai      gpt-4o-mini
  openai      gpt-3.5-turbo
  mistral     mistral-large-latest
  mistral     mistral-small-latest
  mistral     open-mistral-7b
  anthropic   claude-haiku-4-5  <-- default FAST model (extraction)
  anthropic   claude-sonnet-4-5  <-- default SMART model (report)
  anthropic   claude-opus-4-5
  anthropic   claude-3-haiku-20240307
  ollama      llama3.2
  ollama      llama3.2:1b
  ollama      mistral
  ollama      deepseek-r1:1.5b
  ollama      deepseek-r1:7b
  ollama      phi3
  ollama      phi3:mini
  ollama      gemma3
  ollama      gemma3:4b


## Cell 3 — Configure Your Run

This is the only cell you need to edit before running.

- **`COMPANY_URL`** and **`COMPANY_NAME`** — the company you want to research
- **`FAST_MODEL_OVERRIDE`** — used for all extraction steps (pick anything fast and cheap)
- **`SMART_MODEL_OVERRIDE`** — used for the final report (pick anything capable)
- **`MAX_COMPETITORS`** — how many competitors to find and analyse (3–5 is a good range)

To see all available models and which provider they use, run Cell 2.  
To change the defaults permanently, edit `FAST_MODEL` and `SMART_MODEL` in `config.py`.

In [3]:
# ----------------------------------------------------------------
# CONFIGURE HERE
# ----------------------------------------------------------------

COMPANY_URL  = "https://www.stripe.com"   # <-- change to any company URL
COMPANY_NAME = "Stripe"                    # <-- company name (used for search)

FAST_MODEL_OVERRIDE  = "claude-haiku-4-5"   # used for extraction (LLM calls 1, 2, 3)
SMART_MODEL_OVERRIDE = "claude-sonnet-4-5"  # used for report generation (LLM call 4)

MAX_COMPETITORS = 4   # how many competitors to find and analyse (3-5 recommended)

# ----------------------------------------------------------------
print(f"Target company : {COMPANY_NAME}")
print(f"URL            : {COMPANY_URL}")
print(f"Fast model     : {FAST_MODEL_OVERRIDE}")
print(f"Smart model    : {SMART_MODEL_OVERRIDE}")
print(f"Max competitors: {MAX_COMPETITORS}")

Target company : Stripe
URL            : https://www.stripe.com
Fast model     : claude-haiku-4-5
Smart model    : claude-sonnet-4-5
Max competitors: 4


## Cell 4 — Run the Pipeline

This cell runs all 7 steps and prints progress as it goes.  
Typical run time: **30–90 seconds** depending on how many competitors are found.

In [4]:
from report import run_competitor_intelligence

intelligence_report = run_competitor_intelligence(
    company_url=COMPANY_URL,
    company_name=COMPANY_NAME,
    fast_model=FAST_MODEL_OVERRIDE,
    smart_model=SMART_MODEL_OVERRIDE,
    max_competitors=MAX_COMPETITORS,
)


 Competitor Intelligence Report for: Stripe

[Step 1] Checking and scraping main company website...
  [scraper] Scraping: https://www.stripe.com

[Step 2] Extracting main company profile...
  [step] Extracting company profile...
    [llm] anthropic | claude-haiku-4-5

  Profile extracted:
COMPANY NAME: Stripe

INDUSTRY: Financial Technology / Payment Processing

PRODUCT OR SERVICE: Stripe provides comprehensive financial infrastructure that enables businesses to accept payments, offer financial services, and implement custom revenue models globally. The platform supports online and i...

[Step 3] Searching for competitors via Tavily...
  [searcher] Searching: 'top direct competitors of Stripe in Financial Technology / Payment Processing'
  [searcher] Retrieved 10 search results.

[Step 4] Identifying and ranking competitors...
  [step] Extracting and ranking real competitors from search results...
    [llm] anthropic | claude-haiku-4-5
  [!] Removed: Stax (https://www.stax.com)
  [ste

## Cell 5 — Display the Report

In [5]:
from IPython.display import Markdown, display
display(Markdown(intelligence_report))

# Competitive Intelligence Report: Stripe

## 1. Company Overview

Stripe is a leading financial technology company that provides comprehensive payment infrastructure for businesses of all sizes. The platform enables companies to accept payments globally across 135+ currencies and payment methods, while offering an integrated suite of financial services including billing, card issuing, fraud detection, and money movement capabilities.

Stripe processes $1.9 trillion in annual payment volume with 99.999% uptime and serves 50% of Fortune 100 companies. The company positions itself as the financial infrastructure layer for ambitious businesses, supporting any business model "from first to billionth transaction." With products like Stripe Billing (managing 200M+ active subscriptions), Stripe Connect (embedded payments and marketplace solutions), and Stripe Radar (advanced fraud detection), the platform is designed as an integrated ecosystem where tools work seamlessly together.

Stripe's pricing model is usage-based with tiered plans, charging per-transaction fees. The company targets a broad spectrum—from startups to enterprises—particularly focusing on e-commerce, SaaS, marketplaces, and subscription-based businesses. Its tone is distinctly enterprise and developer-focused, emphasizing reliability, scale, and comprehensive capabilities for businesses with complex financial infrastructure needs.

## 2. Competitor Profiles

### PayPal
PayPal is a digital payment platform serving both consumers and businesses globally. While it offers merchant services, PayPal's strength lies in its consumer-facing ecosystem that includes Venmo, buy-now-pay-later options (Pay in 4, Pay Monthly), cryptocurrency transactions, high-yield savings accounts (3.50% APY), and cash back rewards (up to 7%). The platform operates on a freemium model with free consumer accounts and merchant processing fees. PayPal positions itself as an inclusive, consumer-first financial technology leader that makes commerce accessible and rewarding for everyday users while providing enterprise-grade solutions. Its all-in-one approach combines payments, money transfer, financing, rewards, savings, and crypto in a single app.

### Adyen
Adyen is an enterprise-focused payment processing platform built specifically for global businesses. The company provides an end-to-end financial technology solution supporting 150+ currencies and 200+ local payment methods through a single unified platform. Adyen processes €1.4 trillion annually and holds US, UK, and EU banking licenses, giving it unique regulatory positioning. Unlike competitors who have assembled platforms through acquisitions, Adyen built its entire infrastructure in-house on modern technology, resulting in 99.999% historical uptime. The platform supports payments, payouts, data insights, and embedded financial products (accounts, card issuing, capital) through a single API. Adyen targets global enterprises across retail, travel, digital media, SaaS, marketplaces, and financial services, positioning itself as a premium, reliability-focused solution for companies requiring enterprise-grade infrastructure.

### Braintree
Braintree, owned by PayPal, is an enterprise payment processing platform that provides end-to-end checkout experiences and payment orchestration. The platform processes $1.53 trillion in annual payment volume and 25 billion transactions, supporting multiple payment methods including PayPal, Venmo, Apple Pay, Google Pay, and local payment methods across 200+ markets. Braintree offers global payouts to 200+ markets in 50+ currencies and emphasizes adaptive fraud management. With custom enterprise pricing and Level 1 PCI DSS compliance, Braintree positions itself as a developer-focused, enterprise-grade solution backed by two decades of payments experience. Its key differentiator is payment orchestration capabilities and seamless integration with PayPal's ecosystem.

### Square
Square is a comprehensive business operating system that extends far beyond payment processing. The platform serves 4+ million sellers globally, from solo entrepreneurs to large international chains, with a focus on small to mid-market businesses. Square offers payment processing hardware (terminals, readers), point of sale systems tailored for specific industries (restaurants, retail, appointments), inventory management, staff scheduling, payroll, banking services (checking, savings, loans, credit cards), customer loyalty programs, and online ordering capabilities. Square positions itself as "the only platform of its kind" serving businesses of all sizes, emphasizing its 15-year history of democratizing payment acceptance. The company takes an inclusive, all-in-one approach focused on operational management rather than just payment infrastructure.

## 3. Competitive Matrix

| Dimension | Stripe | PayPal | Adyen | Braintree | Square |
|-----------|--------|--------|-------|-----------|--------|
| **Product/Service** | Comprehensive financial infrastructure platform | Digital payment platform with consumer financial services | End-to-end enterprise payments platform | Enterprise payment processing and orchestration | All-in-one business operating system |
| **Target Customers** | Startups to Fortune 100 enterprises; e-commerce, SaaS, marketplaces, subscriptions | Individual consumers, small businesses, online retailers, global enterprises | Global enterprises across retail, travel, digital media, SaaS, marketplaces, financial services | Enterprise businesses and high-volume merchants | Solo entrepreneurs to large chains; food & beverage, retail, beauty, services, healthcare |
| **Pricing Model** | Usage-based per-transaction with tiered plans | Freemium (free consumer accounts + merchant processing fees) | Usage-based (transaction-based with fixed + payment method fees) | Enterprise custom pricing | Unknown (likely usage-based) |
| **Key Features** | Global payments (135+ currencies), Stripe Billing (200M+ subscriptions), card issuing, fraud detection (Radar), embedded payments (Connect), analytics (Sigma) | Multiple payment options, cash back rewards (7%), BNPL, high-yield savings (3.50% APY), crypto, money transfer, Venmo | Unified platform for payments/payouts/data, 150+ currencies, 200+ payment methods, embedded financial products, 99.999% uptime, banking licenses | Multi-payment methods (PayPal, Venmo, wallets), payment orchestration, adaptive fraud management, global payouts (200+ markets, 50+ currencies), PCI DSS Level 1 | Payment hardware/software, POS systems, inventory management, scheduling, payroll, banking services, loyalty programs, online ordering |
| **Unique Selling Points** | $1.9T payment volume, 99.999% uptime, 50% of Fortune 100, integrated ecosystem, supports any business model, professional services | All-in-one consumer app (payments + BNPL + rewards + savings + crypto), trusted brand, global reach, consumer-first approach | Entirely in-house platform, €1.4T volume, US/UK/EU banking licenses, single centralized stack, enterprise reliability | $1.53T volume, 25B transactions, PayPal ecosystem integration, payment orchestration, 20 years experience | 4M+ sellers, 15-year history, democratized payments, comprehensive business operations beyond payments |
| **Tone & Positioning** | Enterprise and developer-focused; reliability, scale, comprehensive infrastructure for ambitious businesses | Consumer-focused, inclusive, innovative; trusted financial technology leader for everyday users and businesses | Enterprise/premium/global; reliability-focused for large-scale operations | Enterprise/developer-focused; backed by PayPal experience | Enterprise-friendly yet SMB-focused; inclusive all-in-one platform for all business sizes |

## 4. Market Standing

### Market Leadership by Category

**Enterprise Infrastructure & Developer Experience: Stripe (Leader)**
Stripe dominates the enterprise developer-focused segment with its comprehensive, well-documented API ecosystem and integrated financial infrastructure. Serving 50% of Fortune 100 companies with $1.9T in payment volume, Stripe has established itself as the go-to platform for businesses requiring sophisticated, scalable payment infrastructure. Its strength lies in supporting complex business models from inception to massive scale.

**Enterprise Reliability & Global Scale: Adyen (Strong Challenger)**
Adyen competes directly with Stripe in the enterprise segment, processing €1.4T annually with a fully in-house built platform. Its unique advantage is holding US, UK, and EU banking licenses, providing regulatory advantages and direct settlement capabilities. Adyen's 99.999% uptime matches Stripe's reliability, and its focus on the largest global enterprises positions it as a premium alternative for companies prioritizing regulatory compliance and direct banking relationships.

**Consumer Brand & Ecosystem: PayPal (Leader)**
PayPal dominates consumer-facing payments with unmatched brand recognition and a comprehensive consumer financial services ecosystem. Its integration of Venmo, BNPL options, savings accounts, rewards, and cryptocurrency creates a moat in consumer payments that competitors cannot easily replicate. For businesses wanting to tap into PayPal's consumer base, this represents significant value.

**Payment Orchestration & PayPal Integration: Braintree (Niche Leader)**
Braintree occupies a unique position as PayPal's enterprise arm, processing $1.53T annually. Its strength is payment orchestration and seamless integration with PayPal's consumer ecosystem (PayPal, Venmo). For enterprises already invested in PayPal or requiring sophisticated payment routing, Braintree offers specialized capabilities, though it lacks the broader financial infrastructure of Stripe or Adyen.

**SMB Operations & Point-of-Sale: Square (Leader)**
Square leads in the small-to-medium business segment with 4M+ sellers, particularly for businesses requiring physical point-of-sale systems and comprehensive operational management. Its strength is not just payment processing but the entire business operating system—inventory, scheduling, payroll, banking. Square democratized payment acceptance for small businesses and maintains that positioning, though it has expanded upmarket.

### Pricing Competitiveness
All major players use usage-based models, though transparency varies. PayPal's freemium consumer model creates a different dynamic, while Braintree's enterprise custom pricing suggests premium positioning. Square's undisclosed pricing likely reflects its hardware-inclusive model. Stripe and Adyen compete directly on transaction-based pricing, with differentiation coming from value-added services rather than base rates.

### Feature Completeness
**Stripe** offers the most comprehensive integrated financial infrastructure for developers and enterprises, with strong subscription billing, fraud detection, and embedded payments capabilities. **Adyen** matches on core payment processing and adds banking licenses. **PayPal/Braintree** excel in consumer payment methods and orchestration. **Square** leads in operational tools beyond payments. Each has carved distinct feature territories.

### Target Market Reach
- **Broadest Enterprise Penetration**: Stripe (50% of Fortune 100)
- **Largest Consumer Base**: PayPal (hundreds of millions of consumer accounts)
- **Premium Enterprise Focus**: Adyen (selective, high-volume merchants)
- **SMB Market Leader**: Square (4M+ sellers)
- **Enterprise PayPal Integration**: Braintree (specialized niche)

## 5. Strategic Recommendations

### 1. Aggressively Counter Adyen's Banking License Advantage
**Gap Identified**: Adyen holds US, UK, and EU banking licenses, providing direct settlement capabilities and regulatory advantages that Stripe lacks. This creates a competitive disadvantage when selling to highly regulated enterprises or those requiring direct banking relationships.

**Recommendation**: Stripe should either pursue strategic banking licenses in key markets or develop partnerships with licensed banks that provide equivalent capabilities under Stripe's brand. Alternatively, create a clear narrative around why Stripe's partnership model provides superior flexibility and innovation speed compared to being a regulated bank. Consider acquiring a small bank or payment institution to gain licensing in strategic markets.

### 2. Develop a Consumer-Facing Payment Brand to Compete with PayPal
**Gap Identified**: PayPal's consumer brand recognition and ecosystem (including Venmo, BNPL, rewards, savings) creates a powerful moat. Merchants choose PayPal not just for infrastructure but to access its consumer base. Stripe has no consumer-facing brand, limiting its ability to drive consumer preference at checkout.

**Recommendation**: Launch a Stripe-branded consumer payment option that merchants can offer at checkout, similar to how "Pay with PayPal" works. This could include a Stripe digital wallet, BNPL options, or rewards program. Leverage Stripe's existing merchant relationships to build network effects. Alternatively, acquire or partner with an existing consumer fintech brand to rapidly gain consumer mindshare.

### 3. Expand Beyond Developer-First to Business-User-First Tools
**Gap Identified**: Square's success with 4M+ sellers demonstrates strong demand for business operating systems that extend beyond payment infrastructure. Stripe's developer-first approach creates friction for non-technical business users who need operational tools (inventory, scheduling, customer management).

**Recommendation**: Develop or acquire a suite of no-code/low-code business operation tools that complement Stripe's payment infrastructure. Create industry-specific packages (e.g., "Stripe for Restaurants," "Stripe for Retail") that bundle payments with operational tools. This would allow Stripe to compete for SMB market share while maintaining its enterprise positioning. Consider white-labeling Square-like functionality for Stripe's existing merchant base.

### 4. Create Transparent Pricing Tiers to Combat Enterprise Custom Pricing Pressure
**Gap Identified**: Braintree's enterprise custom pricing and Adyen's negotiated rates create pricing opacity that can disadvantage Stripe in large enterprise deals. While Stripe has tiered pricing, the lack of transparent enterprise pricing creates uncertainty for large prospects.

**Recommendation**: Introduce clearly defined enterprise pricing tiers with published rates and volume discounts. This transparency would differentiate Stripe from competitors using opaque custom pricing and align with Stripe's developer-friendly, transparent culture. Include clear upgrade paths showing exactly when businesses should move between tiers, reducing sales friction and speeding enterprise adoption.

### 5. Strengthen Point-of-Sale and Physical Retail Capabilities
**Gap Identified**: Square dominates physical point-of-sale with purpose-built hardware and software for in-person transactions. While Stripe supports in-person payments, it lacks Square's comprehensive POS ecosystem and industry-specific solutions for physical retail, restaurants, and service businesses.

**Recommendation**: Significantly expand Stripe Terminal with industry-specific POS solutions. Develop or acquire restaurant management systems, retail POS software, and appointment booking tools that integrate seamlessly with Stripe's payment infrastructure. Create hardware partnerships or develop proprietary devices that compete directly with Square's terminals. Target mid-market and enterprise physical retailers who need both sophisticated payment infrastructure and operational tools.

### 6. Build Payment Orchestration Capabilities to Match Braintree
**Gap Identified**: Braintree's payment orchestration capabilities allow enterprises to route transactions across multiple payment providers, optimize costs, and manage complex payment flows. Stripe's single-provider model, while simpler, lacks the flexibility that large enterprises increasingly demand.

**Recommendation**: Develop Stripe Orchestration as a premium product that allows enterprises to route payments across Stripe and other providers through a single integration. This would position Stripe as the control layer even when merchants use multiple payment processors, preventing displacement by orchestration-focused competitors. Include intelligent routing based on cost, success rates, and business rules.

### 7. Expand Embedded Finance Offerings to Compete with Adyen's Financial Products
**Gap Identified**: Adyen offers embedded financial products including accounts, card issuing, and capital under customer brands through a single platform. While Stripe has card issuing and some embedded finance capabilities, Adyen's unified approach to launching financial products is more comprehensive.

**Recommendation**: Create "Stripe Financial Products Suite" that bundles card issuing, banking-as-a-service, lending, and investment products under a single, easy-to-integrate platform. Enable platforms and marketplaces to launch complete financial services under their own brands with minimal technical lift. Partner with licensed banks to provide FDIC insurance and regulatory compliance while Stripe handles the technology layer. This would strengthen Stripe's platform and marketplace positioning against Adyen's embedded finance capabilities.

## Cell 6 — Save the Report to a File (Optional)

In [ ]:
import re
from datetime import datetime

safe_name = re.sub(r'[^\w]', '_', COMPANY_NAME.lower())
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
filename  = f"{safe_name}_competitor_report_{timestamp}.md"

with open(filename, "w", encoding="utf-8") as f:
    f.write(intelligence_report)

print(f"Report saved to: {filename}")

---

## Quick Reference — Model Options

| Provider | Model | Cost | Good for |
|----------|-------|------|----------|
| **Anthropic** | `claude-haiku-4-5` | Paid | Fast extraction (default FAST) |
| **Anthropic** | `claude-sonnet-4-5` | Paid | Strong report generation (default SMART) |
| **Anthropic** | `claude-opus-4-5` | Paid | Most capable — best report quality |
| Groq | `llama-3.1-8b-instant` | Free | Fast extraction alternative |
| Groq | `llama-3.3-70b-versatile` | Free | Report generation alternative |
| Groq | `gemma2-9b-it` | Free | Fast extraction alternative |
| Gemini | `gemini-2.0-flash` | Free | Fast + capable |
| Gemini | `gemini-1.5-pro` | Free | Strong report generation |
| OpenAI | `gpt-4o-mini` | Paid | Fast extraction |
| OpenAI | `gpt-4o` | Paid | Strong report generation |
| Ollama | `llama3.2` | Local | Fully offline, no cost |
| Ollama | `phi3:mini` | Local | Fully offline, no cost |
